In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
print("Pandas version:", pd.__version__)
print("Analytics environment is ready.")

# Wellness Services Exploratory Data Analysis

In [ ]:
from pathlib import Path

def find_raw_data_dir(start: Path = Path.cwd()) -> Path:
    for directory in (start, *start.parents):
        candidates = (
            directory / "data" / "raw",
            directory / "data-analytics" / "data" / "raw",
        )
        for candidate in candidates:
            if candidate.is_dir():
                return candidate
    raise FileNotFoundError("Could not locate data-analytics/data/raw")

RAW_DATA_DIR = find_raw_data_dir()

In [ ]:
print([file.name for file in sorted(RAW_DATA_DIR.glob("*.csv"))])

In [ ]:
services = pd.read_csv(RAW_DATA_DIR / "Services.csv"); programmes = pd.read_csv(RAW_DATA_DIR / "Programmes.csv"); programme_services = pd.read_csv(RAW_DATA_DIR / "Programme_Services.csv"); screenings = pd.read_csv(RAW_DATA_DIR / "Screenings.csv"); participations = pd.read_csv(RAW_DATA_DIR / "Participations.csv"); print("All five CSV files loaded successfully.")

In [ ]:
print("Services:", services.shape); print("Programmes:", programmes.shape); print("Programme Services:", programme_services.shape); print("Screenings:", screenings.shape); print("Participations:", participations.shape)

## 1. Initial Service Data Inspection

In [ ]:
services

In [ ]:
services.info()

## 2. Missing Value Analysis

In [ ]:
services.isna().sum().to_frame(name="missing_count")

## 3. Service Identifier Quality

In [ ]:
print("Duplicate service IDs:", services["service_id"].duplicated().sum()); print("Duplicate service codes:", services["code"].duplicated().sum()); print("Unique service IDs:", services["service_id"].nunique()); print("Unique service codes:", services["code"].nunique())

In [ ]:
assert services["service_id"].notna().all(); assert services["service_id"].is_unique; assert services["code"].notna().all(); assert services["code"].is_unique; print("Identifier validation passed.")

### Identifier Format Validation

In [ ]:
invalid_service_ids = services[~services["service_id"].str.match(r"^SRV-[A-Z0-9]+$", na=False)]; invalid_service_codes = services[~services["code"].str.match(r"^[a-z][a-z0-9_]*$", na=False)]; print("Invalid service IDs:", len(invalid_service_ids)); print("Invalid service codes:", len(invalid_service_codes))

In [ ]:
assert invalid_service_ids.empty; assert invalid_service_codes.empty; print("Identifier formats are consistent.")

## 4. Service Definitions and Categories

In [ ]:
services[["service_id", "code", "name", "category", "default_unit", "active"]]

In [ ]:
services["category"].value_counts().to_frame(name="service_count")

In [ ]:
services["default_unit"].value_counts().to_frame(name="service_count")

### Service Category Distribution

In [ ]:
plt.figure(figsize=(7, 4)); sns.countplot(data=services, x="category", order=services["category"].value_counts().index, hue="category", palette={"screening": "#172554", "fitness": "#DC2626"}, legend=False); plt.title("Wellness Services by Category"); plt.xlabel("Service Category"); plt.ylabel("Number of Services"); plt.tight_layout(); plt.show()

## 5. Programme-Service Relationship Validation

In [ ]:
programme_services

In [ ]:
orphan_service_ids = sorted(set(programme_services["service_id"]) - set(services["service_id"])); print("Service IDs not found in Services.csv:", orphan_service_ids)

In [ ]:
orphan_programme_ids = sorted(set(programme_services["programme_id"]) - set(programmes["programme_id"])); print("Programme IDs not found in Programmes.csv:", orphan_programme_ids)

In [ ]:
assert not orphan_service_ids; assert not orphan_programme_ids; assert programme_services["programme_service_id"].is_unique; assert not programme_services.duplicated(subset=["programme_id", "service_id"]).any(); print("Programme-service relationship validation passed.")

✅ test(analytics): validate programme-service relationships

### Join Services to Programmes

In [ ]:
programme_service_join = programme_services.merge(services, on="service_id", how="left", validate="many_to_one", indicator="service_join_status"); programme_service_join

In [ ]:
programme_service_join["service_join_status"].value_counts()

In [ ]:
programme_service_join = programme_service_join.merge(programmes[["programme_id", "name", "programme_type", "status", "target_participants"]].rename(columns={"name": "programme_name"}), on="programme_id", how="left", validate="many_to_one", indicator="programme_join_status"); programme_service_join

In [ ]:
assert len(programme_service_join) == len(programme_services); assert programme_service_join["service_join_status"].eq("both").all(); assert programme_service_join["programme_join_status"].eq("both").all(); print("All programme-service joins succeeded without losing rows.")

## 6. Downstream Screening Join Validation

In [ ]:
screening_bridge_join = screenings.merge(programme_services[["programme_service_id", "programme_id", "service_id"]], on="programme_service_id", how="left", validate="many_to_one", indicator="programme_service_join_status"); screening_bridge_join.head()

In [ ]:
screening_service_join = screening_bridge_join.merge(services[["service_id", "code", "name", "category", "default_unit"]].rename(columns={"name": "service_name"}), on="service_id", how="left", validate="many_to_one", indicator="service_join_status"); screening_service_join.head()

In [ ]:
print("Original screening rows:", len(screenings)); print("Joined screening rows:", len(screening_service_join)); print("Unmatched programme services:", screening_service_join["programme_service_join_status"].ne("both").sum()); print("Unmatched services:", screening_service_join["service_join_status"].ne("both").sum())

In [ ]:
assert len(screening_service_join) == len(screenings); assert screening_service_join["programme_service_join_status"].eq("both").all(); assert screening_service_join["service_join_status"].eq("both").all(); print("All downstream screening joins succeeded without losing rows.")

## 7. Conclusion and Recommendations

The service data is correctly structured for connecting different parts of the Pulse80 system. Each wellness service has its own unique service_id and code. There are no missing or duplicated service identifiers.

The service IDs also connect successfully to Programme_Services.csv. This means Pulse80 can identify which services belong to a particular wellness programme. The screening records can then connect to those programme services using programme_service_id.

All 144 screening records joined successfully to their corresponding services. No records were lost, and no screening referenced an unknown service. Therefore, the current relationship between services, programmes and screenings is working correctly.

However, the information describing some services is still limited. The service table tells us the service’s name, category and default unit, but it does not fully explain what data the service produces or how that data should be interpreted.

For example, stress and fitness both use score as their unit. The data does not explain:

What the score is out of
How the score is calculated
What a low score means
What a high score means
Which assessment method produced the score

This would make it difficult for Pulse80 to produce reliable comparisons, risk classifications and dashboard insights from those scores.

The existing categories also work for the six services currently available, but only screening and fitness are represented. This may be insufficient when Pulse80 adds services such as counselling, mental-health support, laboratory testing, wellness education and preventive care.

Overall, the service catalogue is suitable for connecting the current datasets. Its identifiers and table relationships are reliable. However, the service definitions need more detail before the catalogue can confidently support larger-scale analytics and reporting.

The recommended next step is to keep the existing identifiers and relationships, while gradually adding clearer service descriptions, measurement definitions, scoring rules and controlled categories as the Pulse80 service catalogue grows.